# Pricing the Unpriced — Starter Notebook
**NYU CUSP Capstone 2026–2027 · Sponsor: BNBD / Oxcart Assembly**

This notebook gets you from zero to the real research question in about five minutes.
It loads the corridor knowledge base, shows you how the tables join, builds the
people–place–event graph, and ends at the open problem you would be solving.

Put this file in the same folder as the CSVs and run top to bottom.
Requires: `pandas`, `matplotlib`, `networkx`.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, networkx as nx, collections
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

props   = pd.read_csv("properties.csv", low_memory=False)
inc     = pd.read_csv("property_incidents.csv", low_memory=False)
subj    = pd.read_csv("subjects.csv", low_memory=False)
isub    = pd.read_csv("incident_subjects.csv", low_memory=False)
ilink   = pd.read_csv("incident_links.csv", low_memory=False)
rips    = pd.read_csv("registered_ips.csv", low_memory=False)
parcels = pd.read_csv("property_parcels.csv", low_memory=False)
grants  = pd.read_csv("grant_program_matches.csv", low_memory=False)

for name, df in [("properties",props),("incidents",inc),("subjects",subj),
                 ("incident_subjects",isub),("incident_links",ilink),
                 ("registered_ips",rips),("parcels",parcels),("grant_matches",grants)]:
    print(f"{name:<20} {len(df):>7,} rows   {len(df.columns):>3} cols")

## 1. The first thing you need to know

The incident table is **not** 20,308 pieces of curated history. Most of it is
machine-ingested administrative data: 311 complaints, permits, assessments, crime.
The curated archival layer is much smaller and much older.

Separating signal from volume is not a preprocessing chore here. **It is the research problem.**
The proposal calls this the archival-abundance risk: an engine that rewards whichever
property generated the most paperwork has learned nothing about historical significance.

In [ ]:
ADMIN_PREFIXES = ("baltimore:",)
ADMIN_EXACT = {"sdat_assessments", "sdat:owner"}
def is_admin(s):
    s = str(s)
    return s.startswith(ADMIN_PREFIXES) or s in ADMIN_EXACT

inc["layer"] = inc["source"].map(lambda s: "administrative" if is_admin(s) else "curated")
print(inc["layer"].value_counts(), "\n")
print("Curated sources:")
print(inc[inc.layer=="curated"]["source"].value_counts().head(15))

## 2. Evidence grading — the claim record

Every incident is a *claim*, not a fact. The schema carries `evidence_status`,
`date_precision`, `sensitivity`, and `rights`. Coverage is partial and uneven,
which is itself a finding: back-filling and validating this grading is committed capstone work.

In [ ]:
print("evidence_status:\n", inc["evidence_status"].fillna("(ungraded)").value_counts(), "\n")
print("date_precision:\n", inc["date_precision"].fillna("(unset)").value_counts(), "\n")
cur = inc[inc.layer=="curated"]
print(f"curated rows graded: {cur['evidence_status'].notna().sum():,} of {len(cur):,}")
print("\nincident-to-incident link types (the graph already encodes disagreement):")
print(ilink["link_type"].value_counts())

## 3. Where the depth actually is

Documentation is concentrated. A handful of properties carry deep archival research;
most carry only administrative traces. This ranking is where you should start —
it is the labeled spine the index gets built and validated on.

In [ ]:
depth = (cur.groupby("property_id").size().rename("curated_incidents")
           .reset_index().merge(props[["id","address","year_built","assessed_value"]],
                                left_on="property_id", right_on="id", how="left")
           .drop(columns="id").sort_values("curated_incidents", ascending=False))
print(f"properties with >=1 curated incident: {len(depth):,}")
print(f"  >=10: {(depth.curated_incidents>=10).sum():,}    >=25: {(depth.curated_incidents>=25).sum():,}")
print("\nDeepest-documented properties:")
print(depth.head(12).to_string(index=False))

## 4. The temporal shape of the corpus

In [ ]:
inc["year"] = pd.to_numeric(inc["occurred_at"].astype(str).str[:4], errors="coerce")
d = inc[(inc.year>=1700)&(inc.year<=2029)].copy()
d["decade"] = (d.year//10*10).astype(int)
piv = d.pivot_table(index="decade", columns="layer", values="id", aggfunc="count").fillna(0)
ax = piv.plot(kind="bar", stacked=True, figsize=(14,5), width=.85,
              color={"administrative":"#8a8377","curated":"#e8a82d"})
ax.set_yscale("symlog", linthresh=10); ax.set_xlabel(""); ax.set_ylabel("incidents")
ax.set_title("Documented incidents by decade — curated vs administrative")
plt.tight_layout(); plt.show()
print("pre-2000 incidents:", int((d.year<2000).sum()), " | pre-1950:", int((d.year<1950).sum()))

## 5. The graph

This is what the corridor has that a standard hedonic dataset does not:
people, businesses, families and organizations, linked to dated events at addresses.
The committed connectivity test asks whether *narrative* connection carries information
that physical proximity does not.

In [ ]:
print(subj["subject_type"].value_counts(), "\n")
print(isub["relationship"].value_counts().head(10))

edges = (isub.merge(inc[["id","property_id"]], left_on="property_incident_id", right_on="id")
             .merge(subj[["id","name","subject_type"]], left_on="subject_id", right_on="id",
                    suffixes=("_inc","_subj")))
G = nx.Graph()
for _, r in edges.iterrows():
    p = f"prop:{r.property_id}"; s = f"subj:{r.subject_id}"
    G.add_node(p, kind="property"); G.add_node(s, kind="subject", name=r["name"])
    G.add_edge(p, s, relationship=r["relationship"])
print(f"\ngraph: {G.number_of_nodes():,} nodes / {G.number_of_edges():,} edges")
print(f"connected components: {nx.number_connected_components(G):,}")
comp = max(nx.connected_components(G), key=len)
print(f"largest component: {len(comp):,} nodes")

## 6. A first, naive connectivity score

Two properties are *narratively connected* if they share a subject: the same person,
family, or business appears at both. Below is the crudest possible version.
Improving this — weighting by relationship type, evidence status, and time overlap —
is exactly the Connectivity term of the Provenance Index.

In [ ]:
subj_to_props = collections.defaultdict(set)
for _, r in edges.iterrows():
    subj_to_props[r.subject_id].add(r.property_id)

pair = collections.Counter()
for sid, ps in subj_to_props.items():
    ps = sorted(ps)
    for i in range(len(ps)):
        for j in range(i+1, len(ps)):
            pair[(ps[i], ps[j])] += 1

print(f"property pairs sharing >=1 subject: {len(pair):,}")
addr = props.set_index("id")["address"].to_dict()
print("\nMost narratively connected property pairs:")
for (a,b), n in pair.most_common(10):
    print(f"  {n:>3} shared subjects   {addr.get(a,a)}  <->  {addr.get(b,b)}")

## 7. Your starting line

What exists: the evidence layer, the graph, the parcel linkage, program matches,
and the outcome fields (`assessed_value`, `last_sale_price`, `last_sale_date`).

What does not exist yet, and what the capstone commits to building:

1. **A pre-specified evidence score** — verifiability, density, salience, distinctiveness,
   connectivity — constructed *blind to prices*, with a frozen coding protocol.
2. **Held-out validation** — does that score add explanatory power beyond building age,
   size, location, and time? Spatially blocked holdouts, not random splits.
3. **The connectivity test** — does narrative connection beat physical proximity?
4. **The honest failure rule** — if the score does not clear its threshold,
   the public tool ships the evidence layer with **no** dollar estimate.

Two cautions the proposal commits to in writing, and you should hold from day one:

- **Archival abundance is not significance.** Never treat sparse documentation as "no history."
- **No temporal leakage.** A white paper written in 2026 cannot inform a 2005 sale price.
  Reconstruct what was knowable *as of* the outcome date.

In [ ]:
out = props[["id","address","year_built","assessed_value","last_sale_price","last_sale_date",
             "latitude","longitude","historic_district","vacancy_indicator"]].copy()
out = out.merge(cur.groupby("property_id").size().rename("curated_incidents"),
                left_on="id", right_index=True, how="left").fillna({"curated_incidents":0})
print("Modeling frame preview:")
print(out.sort_values("curated_incidents", ascending=False).head(10).to_string(index=False))
print(f"\nrows: {len(out):,} | with coords: {out.latitude.notna().sum():,} | with sale price: {out.last_sale_price.notna().sum():,}")